# colab7 stress test 2 — targeted reruns + structural test

Follow-up to `colab7_stress_test.ipynb`. Three things:

1. **ReLU-aware error metric** applied throughout. Negative weights are clipped to 0 before comparing to target, since the network's forward pass uses `relu(w_i)`.
2. **Targeted reruns at more epochs** — only the seeds/configurations that previously failed.
   - Seed robustness: rerun seeds 3, 8, 11, 12, 17, 18 for DESIRE='01011' at 50 epochs
   - Partial training: rerun 50% (seeds 0, 1) and 25% (seeds 2, 4) at 100 epochs
3. **Length 8 with a non-periodic target** (DESIRE='00110101') to isolate whether previous length-8 failure was caused by periodicity or by depth.

Architecture: identical to colab6 / colab7 (Design B, {0,1} alphabet, init in [0,1], SGD lr=0.1).

## Setup

In [ ]:
import os

os.chdir('/content')
!rm -rf /content/thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git

os.chdir('/content/thesis-edit-distance-nn')

!ls sampledata/

In [ ]:
!pip install tensorflow ml_dtypes --upgrade

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import random
import time
import os
import math
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Activation, Input, add, Lambda, Reshape, concatenate, Flatten

## Architecture (identical to colab6 Design B)

In [ ]:
def transform_seqs_to_input(seqA, seqB):
    matching_pairs = []
    input_length_x = 0

    matching_pairs.append([int(seqA[0]), int(seqB[0])])
    if len(seqA) == 1 and len(seqB) == 1:
        return matching_pairs
    else:
        input_length_x = len(seqA)
        match_layers_i = (input_length_x * 2) - 1

    start_i = 1
    end_i = 2

    for l in range(match_layers_i):
        if l < input_length_x - 1:
            i, j = [*reversed(range(0, end_i))], [*range(0, end_i)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            end_i += 1
        else:
            i, j = [*reversed(range(start_i, input_length_x))], [*range(start_i, input_length_x)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            start_i += 1
            if start_i > len(seqB):
                break

    return matching_pairs


def transform_input_for_generate(input):
    x = []
    y = []
    for pair in input:
        x.append(pair[0])
        y.append(pair[1])
    return [x, y]

In [ ]:
def matching_module():
    epsilon = 1

    model = Sequential()
    model.add(Dense(units=2, activation='relu', use_bias=True, input_shape=(2,)))
    model.add(Dense(units=2, activation='relu', use_bias=True))
    model.add(Dense(units=1, activation='relu', use_bias=True))

    w1 = model.layers[0].get_weights()
    w1[0][0][0], w1[0][0][1] = 1.0, -1.0
    w1[0][1][0], w1[0][1][1] = -1.0, 1.0
    w1[1][0], w1[1][1] = 0, 0
    w2 = model.layers[1].get_weights()
    w2[0][0][0], w2[0][0][1] = 1.0, 1.0
    w2[0][1][0], w2[0][1][1] = 1.0, 1.0
    w2[1][0], w2[1][1] = epsilon, -1 * epsilon
    w3 = model.layers[2].get_weights()
    w3[0][0][0], w3[0][1][0] = (1.0/epsilon), -1.0 * (1.0/epsilon)
    w3[1][0] = -1

    model.layers[0].set_weights(w1)
    model.layers[1].set_weights(w2)
    model.layers[2].set_weights(w3)

    model.trainable = False
    return model

In [ ]:
def min_module(i, j, k):
    input = Input(shape=(2,))
    x = Dense(2, activation='relu', use_bias=True)(input)
    combined = concatenate([x, input])

    layer_name = 'result_pixel_' + str(i) + str(j) + '_' + str(k)
    z = Dense(1, activation='relu', use_bias=True, name=layer_name)(combined)
    model = Model(inputs=input, outputs=z)

    w1 = model.layers[1].get_weights()
    w1[0][0], w1[0][1] = [-1.0, 1.0], [1.0, -1.0]
    w2 = model.layers[3].get_weights()
    w2[0][0], w2[0][1], w2[0][2], w2[0][3] = -0.5, -0.5, 0.5, 0.5

    model.layers[1].set_weights(w1)
    model.layers[3].set_weights(w2)

    model.trainable = False
    return model


def minimum(i, j):
    input = Input(shape=(3,))
    comp1_pair = Lambda(lambda x: x[:, :2], output_shape=(2,))(input)
    comp2_input = Lambda(lambda x: x[:, 2:], output_shape=(1,))(input)

    m = min_module(i, j, 1)(comp1_pair)
    comp2_pair = concatenate([comp2_input, m])
    output = min_module(i, j, 2)(comp2_pair)

    model = Model(inputs=input, outputs=output)
    model.trainable = False
    return model

In [ ]:
def align_model_for_N(seq_length_x, seq_length_y, matching_pair_number):
    input = Input(shape=(2, matching_pair_number), name='input')

    y = Lambda(lambda t: t[:, 1, :], output_shape=(matching_pair_number,))(input)
    x = Lambda(lambda t: t[:, 0, :], output_shape=(matching_pair_number,))(input)

    out = {}
    start_i = 0
    step = 2
    for i in range(seq_length_y):
        a = start_i
        layername = 'for_gen_dense_' + str(i + 1)
        # === DESIGN B ===
        ones_input = Lambda(lambda t, a=a: tf.ones_like(t[:, a:a+1]), output_shape=(1,))(y)
        z = Dense(1, activation='relu', name=layername, use_bias=False)(ones_input)
        out[layername] = z
        start_i += step
        step += 1

    pair_i = 1
    calc_layer = (seq_length_x * 2) - 1
    test_dict = {}

    y_dense_layer_name = 'for_gen_dense_1'
    densed_y = out[y_dense_layer_name]
    x_char = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(x)

    debug_name = 'matching_debug_1'
    pair_11 = concatenate([x_char, densed_y], name=debug_name)

    ext_gaps = Dense(2, activation='relu', name='first_calc_gap_layer')(pair_11)

    min1 = min_module(1, 1, 1)(ext_gaps)
    matching1 = matching_module()(pair_11)
    combined = concatenate([min1, matching1])
    z = min_module(1, 1, 2)(combined)
    result_pixel_11 = concatenate([ext_gaps, z], name='input_pixel_1_1')

    pair_i = 2

    if seq_length_x == 1 and seq_length_y == 1:
        output = z
        return Model(inputs=input, outputs=output)
    else:
        test_dict['input_pixel_1_1'] = result_pixel_11
        test_dict['result_pixel_1_1'] = z

        comp_i_val, comp_j_val = 1, 2
        start_sentinel, end_sentinel = 1, 2
        unbalance_flag = True

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_length_x - 1:
                comp_i_val, comp_j_val = start_sentinel, end_sentinel
                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_i = comp_i_val
                        y_dense_layer_name = 'for_gen_dense_' + str(y_i)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)

                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        if comp_i_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        elif comp_j_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        else:
                            previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                            previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                            previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                            before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                if end_sentinel + 1 <= seq_length_x:
                    end_sentinel += 1

            else:
                start_sentinel += 1
                comp_i_val, comp_j_val = start_sentinel, end_sentinel

                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_dense_layer_name = 'for_gen_dense_' + str(comp_i_val)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)
                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                        previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                        previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                        before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                    if start_sentinel == end_sentinel:
                        return Model(inputs=input, outputs=result_pixel)

In [ ]:
def set_weight_for_debug(model, seq_len_x, seq_len_y, matching_pair):
    w = model.get_layer('first_calc_gap_layer').get_weights()
    w[0][0][0], w[0][0][1] = 0, 0
    w[0][1][0], w[0][1][1] = 0, 0
    w[1][0], w[1][1] = 2, 2
    model.get_layer('first_calc_gap_layer').set_weights(w)
    model.get_layer('first_calc_gap_layer').trainable = False

    if seq_len_x > 1:
        calc_layer = (seq_len_x * 2) - 1
        comp_i, comp_j = 1, 2
        start_sentinel, end_sentinel = 1, 2

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_len_x - 1:
                comp_i, comp_j = start_sentinel, end_sentinel
                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        if comp_i == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        elif comp_j == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        else:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, 0

                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)
                if end_sentinel + 1 <= seq_len_x:
                    end_sentinel += 1
            else:
                start_sentinel = start_sentinel + 1
                comp_i, comp_j = start_sentinel, end_sentinel

                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                        w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                        w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                        w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                        w[1][0], w[1][1], w[1][2] = 1, 1, 0
                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)


def froozen_align_model(model):
    for layer in model.layers:
        if 'for_gen_dense' in layer.name:
            layer.trainable = True
        else:
            layer.trainable = False

## Helpers (with ReLU-aware error metric)

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]


def generate_csv(desire):
    n = len(desire)
    csv_path = f'/content/thesis-edit-distance-nn/sampledata/desired_length_{n}_levenshtein_{desire}.csv'
    lines = []
    for i in range(2**n):
        s = format(i, f'0{n}b')
        d = levenshtein(s, desire)
        lines.append(f'{s},{desire},{d}')
    with open(csv_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    return csv_path, lines


def relu_aware_max_err(final_weights, target_weights):
    """Apply ReLU to weights before comparing to target.
    The forward pass uses relu(w_i), so a slightly negative w_i is functionally
    equivalent to 0. Naive abs(w - target) overcounts in that case.
    """
    relu_w = [max(0.0, w) for w in final_weights]
    return max(abs(rw - t) for rw, t in zip(relu_w, target_weights))

In [ ]:
def training(desire, csv_path, epochs=20, learning_rate=0.1, seed=None,
             subset_fraction=None, verbose=True):
    """Vanilla SGD training for Design B. Returns both naive and ReLU-aware max error."""
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)

    LEN = len(desire)

    all_lines = []
    with open(csv_path, 'r') as f:
        for line in f:
            all_lines.append(line.rstrip('\n'))

    if subset_fraction is not None and subset_fraction < 1.0:
        k = max(1, int(len(all_lines) * subset_fraction))
        lines = random.sample(all_lines, k)
    else:
        lines = all_lines

    target_weights = [int(c) for c in desire]
    if verbose:
        print(f'=== DESIRE={desire} | seed={seed} | samples={len(lines)}/{len(all_lines)} | epochs={epochs} ===')

    sp = lines[0].split(',')
    x, y = sp[0], sp[1]
    pairs = transform_seqs_to_input(x, y)
    SEQ_LEN_X = len(x)
    SEQ_LEN_Y = len(y)
    PAIRS_LEN = len(pairs)

    model = align_model_for_N(SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    set_weight_for_debug(model, SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    froozen_align_model(model)

    # Oracle check
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        w = model.get_layer(lname).get_weights()
        w[0][0][0] = float(target_weights[i])
        model.get_layer(lname).set_weights(w)

    all_correct = True
    for line in all_lines:
        sp = line.split(',')
        xs, ys, true_score = sp[0], sp[1], int(sp[2])
        inp = transform_seqs_to_input(xs, ys)
        inp = transform_input_for_generate(inp)
        inp = tf.constant([inp], dtype=tf.float32)
        pred = float(model(inp, training=False)[0][0])
        if abs(pred - true_score) > 0.01:
            if verbose:
                print(f'  ORACLE MISMATCH: x={xs} y={ys} true={true_score} pred={pred:.4f}')
            all_correct = False
    if verbose:
        print(f'  oracle: {"PASS" if all_correct else "FAIL"}')

    init_trained_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        w = model.get_layer(lname).get_weights()
        w[0][0][0] = random.uniform(0.0, 1.0)
        model.get_layer(lname).set_weights(w)
        init_trained_weights.append(float(w[0][0][0]))

    progress_weights = [list(init_trained_weights)]
    progress_losses = []

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.MeanSquaredError()

    t0 = time.time()
    for epoch in range(epochs):
        loss = tf.Variable(0.0, name='loss')
        with tf.GradientTape() as tape:
            for line in lines:
                sp = line.split(',')
                xs, ys, true_score = sp[0], sp[1], int(sp[2])
                inp = transform_seqs_to_input(xs, ys)
                inp = transform_input_for_generate(inp)
                inp = tf.constant([inp], dtype=tf.float32)
                logit = model(inp, training=True)
                loss = loss + loss_fn(true_score, logit)
            batch_loss = loss / len(lines)
            grads = tape.gradient(batch_loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

        progress_losses.append(float(batch_loss.numpy()))
        cur = []
        for i in range(LEN):
            lname = 'for_gen_dense_' + str(i + 1)
            cur.append(float(model.get_layer(lname).get_weights()[0][0][0]))
        progress_weights.append(cur)
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f'epoch {epoch:3d} | loss = {float(batch_loss.numpy()):.6f} | weights = {[round(w, 4) for w in cur]}')
    wall_time = time.time() - t0

    final_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        final_weights.append(float(model.get_layer(lname).get_weights()[0][0][0]))

    naive_max_err = max(abs(f - t) for f, t in zip(final_weights, target_weights))
    relu_max_err = relu_aware_max_err(final_weights, target_weights)
    naive_pass = naive_max_err < 0.05
    relu_pass = relu_max_err < 0.05

    if verbose:
        print(f'\nfinal     weights: {[round(w, 4) for w in final_weights]}')
        print(f'relu(w)   weights: {[round(max(0.0, w), 4) for w in final_weights]}')
        print(f'target    weights: {target_weights}')
        print(f'naive max_err: {naive_max_err:.6f} | {"PASS" if naive_pass else "FAIL"}')
        print(f'relu  max_err: {relu_max_err:.6f} | {"PASS" if relu_pass else "FAIL"}')
        print(f'wall time: {wall_time:.1f}s')

    return {
        'desire': desire,
        'seed': seed,
        'epochs': epochs,
        'subset_fraction': subset_fraction,
        'target_weights': target_weights,
        'init_weights': init_trained_weights,
        'final_weights': final_weights,
        'progress_weights': progress_weights,
        'progress_losses': progress_losses,
        'naive_max_err': naive_max_err,
        'relu_max_err': relu_max_err,
        'naive_pass': naive_pass,
        'relu_pass': relu_pass,
        'wall_time': wall_time,
        'n_samples': len(lines),
        'n_total_samples': len(all_lines),
    }

In [ ]:
def plot_training(result, title=''):
    progress_weights = np.array(result['progress_weights'])
    losses = result['progress_losses']
    target = result['target_weights']
    LEN = progress_weights.shape[1]
    epochs = progress_weights.shape[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for w_i in range(LEN):
        axes[0].plot(range(epochs), progress_weights[:, w_i], marker='o', markersize=2,
                     label=f'w_{w_i+1} (target={target[w_i]})', color=color_cycle[w_i % len(color_cycle)])
        axes[0].axhline(target[w_i], color=color_cycle[w_i % len(color_cycle)], linestyle='dotted', alpha=0.5)
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('weight value')
    axes[0].set_title('weight trajectories')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    axes[1].plot(range(1, len(losses) + 1), losses, marker='o', markersize=2, color='red')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('MSE loss')
    axes[1].set_title('training loss')
    axes[1].set_ylim(bottom=0)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Generate CSV files

Length 5 (`'01011'`, 32 samples) and length 8 (`'00110101'`, 256 samples).

In [ ]:
csv_path_01011, _ = generate_csv('01011')
csv_path_00110101, _ = generate_csv('00110101')

print(f'01011:    2^5 = {2**5} samples')
print(f'00110101: 2^8 = {2**8} samples')

---
# Tier 1 — Re-evaluate previous results with ReLU-aware metric

Apply the new metric to the previously logged final weights from `colab7_stress_test.ipynb` (DESIRE='01011', 20 seeds, 20 epochs). No re-training needed.

In [ ]:
# Final weights from previous run (colab7 experiment 1, DESIRE='01011', 20 epochs, seeds 0-19)
previous_01011_finals = {
    0:  [0.016, 0.997, -0.0,   1.0,   0.995],
    1:  [-0.006, 0.976, 0.015, 0.996, 0.999],
    2:  [-0.005, 0.995, 0.03,  0.957, 0.999],
    3:  [0.001, 0.925, 0.055,  1.003, 0.975],
    4:  [0.009, 0.981, -0.002, 0.999, 0.975],
    5:  [0.028, 0.979, -0.005, 1.002, 0.991],
    6:  [-0.004, 0.976, 0.02,  1.0,   0.987],
    7:  [0.016, 0.98,  0.005,  1.0,   1.0],
    8:  [0.01,  0.947, 0.042,  0.994, 0.999],
    9:  [-0.001, 0.991, 0.014, 1.001, 0.988],
    10: [0.006, 0.971, 0.023,  0.998, 0.999],
    11: [0.015, 0.94,  0.049,  0.993, 0.997],
    12: [0.013, 0.947, 0.047,  0.994, 0.995],
    13: [-0.007, 0.986, 0.007, 0.999, 0.965],
    14: [0.002, 0.997, 0.005,  0.988, 1.0],
    15: [-0.0,  0.962, 0.034,  1.0,   0.986],
    16: [-0.007, 0.961, 0.035, 1.0,   0.988],
    17: [0.018, 0.945, 0.038,  1.0,   0.991],
    18: [0.417, -0.01, 0.99,   0.414, 0.988],   # genuine bad minimum
    19: [0.015, 0.953, 0.046,  1.0,   0.996],
}
target_01011 = [0, 1, 0, 1, 1]

print('='*90)
print("Re-evaluating colab7 experiment 1 (DESIRE='01011', 20 epochs) with ReLU-aware metric")
print('='*90)
print(f"{'seed':>4} | {'naive_err':>10} | {'naive':>5} | {'relu_err':>10} | {'relu':>5}")
print('-'*60)
naive_pass = 0
relu_pass = 0
for s in range(20):
    fw = previous_01011_finals[s]
    n_err = max(abs(f - t) for f, t in zip(fw, target_01011))
    r_err = relu_aware_max_err(fw, target_01011)
    n_p = 'PASS' if n_err < 0.05 else 'FAIL'
    r_p = 'PASS' if r_err < 0.05 else 'FAIL'
    if n_err < 0.05: naive_pass += 1
    if r_err < 0.05: relu_pass += 1
    print(f'{s:>4} | {n_err:>10.4f} | {n_p:>5} | {r_err:>10.4f} | {r_p:>5}')

print()
print(f'Naive metric pass rate: {naive_pass}/20')
print(f'ReLU-aware pass rate:   {relu_pass}/20')

---
# Tier 2a — Rerun the failed seeds at 50 epochs

From colab7 experiment 1, these seeds failed at 20 epochs: **3, 8, 11, 12, 17, 18**.

Hypothesis:
- Seeds 3, 8, 11, 12, 17 are under-converged and will pass at 50 epochs.
- Seed 18 is a genuine local minimum and will still fail.

In [ ]:
print('='*80)
print("TIER 2a: Rerun failed seeds for DESIRE='01011' at 50 epochs")
print('='*80)

failed_seeds = [3, 8, 11, 12, 17, 18]
rerun_results = {}

for s in failed_seeds:
    print(f'\n--- seed {s} ---')
    r = training('01011', csv_path_01011, epochs=50, learning_rate=0.1, seed=s, verbose=False)
    rerun_results[s] = r
    print(f"  init   : {[round(w,3) for w in r['init_weights']]}")
    print(f"  final  : {[round(w,3) for w in r['final_weights']]}")
    print(f"  naive max_err: {r['naive_max_err']:.4f} | {'PASS' if r['naive_pass'] else 'FAIL'}")
    print(f"  relu  max_err: {r['relu_max_err']:.4f} | {'PASS' if r['relu_pass'] else 'FAIL'}")

In [ ]:
print('\n' + '='*80)
print('Summary: previous (20 epochs) vs rerun (50 epochs)')
print('='*80)
print(f"{'seed':>4} | {'20ep naive':>10} | {'20ep status':>11} | {'50ep naive':>10} | {'50ep relu':>10} | {'50ep status':>11}")
print('-'*75)
for s in failed_seeds:
    fw_20 = previous_01011_finals[s]
    err_20 = max(abs(f - t) for f, t in zip(fw_20, target_01011))
    r = rerun_results[s]
    status_50 = 'PASS' if r['relu_pass'] else 'FAIL'
    print(f'{s:>4} | {err_20:>10.4f} | {"FAIL":>11} | {r["naive_max_err"]:>10.4f} | {r["relu_max_err"]:>10.4f} | {status_50:>11}')

n_fixed = sum(1 for s in failed_seeds if rerun_results[s]['relu_pass'])
still_failing = [s for s in failed_seeds if not rerun_results[s]['relu_pass']]
print(f'\n{n_fixed}/{len(failed_seeds)} seeds fixed by 50 epochs (using ReLU-aware metric)')
print(f'Still failing: {still_failing}')

# Combined pass rate at 50 epochs (assume previously passing seeds still pass)
originally_passing = [s for s in range(20) if s not in failed_seeds]
combined_pass = len(originally_passing) + n_fixed
print(f'\nProjected pass rate at 50 epochs: {combined_pass}/20')

---
# Tier 2b — Rerun partial training failures at 100 epochs

From colab7 experiment 3 (DESIRE='01011' with subsets), these failed at 50 epochs:
- 50% subset: seeds 0 (max_err=1.72), 1 (max_err=0.40)
- 25% subset: seeds 2 (max_err=0.16), 4 (max_err=1.00)

In [ ]:
print('='*80)
print('TIER 2b: Rerun partial training failures at 100 epochs')
print('='*80)

partial_reruns = []
for frac, label, seeds in [(0.50, '50%', [0, 1]), (0.25, '25%', [2, 4])]:
    print(f'\n--- {label} subset, 100 epochs ---')
    for s in seeds:
        r = training('01011', csv_path_01011, epochs=100, learning_rate=0.1,
                     seed=s, subset_fraction=frac, verbose=False)
        partial_reruns.append((label, s, r))
        print(f"  seed {s} | samples={r['n_samples']}/{r['n_total_samples']}")
        print(f"    final: {[round(w,3) for w in r['final_weights']]}")
        print(f"    naive max_err: {r['naive_max_err']:.4f} | {'PASS' if r['naive_pass'] else 'FAIL'}")
        print(f"    relu  max_err: {r['relu_max_err']:.4f} | {'PASS' if r['relu_pass'] else 'FAIL'}")

In [ ]:
print('\n' + '='*80)
print('Summary: 50 epochs vs 100 epochs (partial training)')
print('='*80)

# Final weights from colab7 exp 3, 50-epoch run (for comparison)
previous_partial_50ep = {
    ('50%', 0): [-0.009, 0.983, 1.719, 0.625, 0.988],
    ('50%', 1): [0.238, 0.595, 0.384, 0.843, 0.932],
    ('25%', 2): [-0.159, 1.0, -0.0, 1.0, 1.0],
    ('25%', 4): [-0.027, -0.003, 0.504, 0.766, 1.002],
}

print(f"{'subset':>6} | {'seed':>4} | {'50ep relu':>10} | {'100ep naive':>11} | {'100ep relu':>11} | {'verdict':>11}")
print('-'*70)
for label, s, r in partial_reruns:
    prev_fw = previous_partial_50ep[(label, s)]
    prev_relu = relu_aware_max_err(prev_fw, target_01011)
    verdict = 'FIXED' if r['relu_pass'] else ('IMPROVED' if r['relu_max_err'] < prev_relu else 'STUCK')
    print(f'{label:>6} | {s:>4} | {prev_relu:>10.4f} | {r["naive_max_err"]:>11.4f} | {r["relu_max_err"]:>11.4f} | {verdict:>11}')

---
# Tier 3 — Length 8 with non-periodic target

Previous run with periodic `'01010101'` failed badly: positions 1–4 converged correctly, positions 5–8 were stuck (one swap, two undecided at 0.5). Hypothesis: periodicity creates many near-degenerate global minima which trap SGD.

**Test:** non-periodic target `'00110101'` (no period < 8). If this PASSES at 50 epochs, periodicity is the structural blocker, not depth.

**Expected wall time:** ~4–5 hours (256 samples × 50 epochs).

In [ ]:
print('='*80)
print("TIER 3: DESIRE='00110101' (length 8, non-periodic, 256 samples, 50 epochs)")
print('='*80)

result_len8_nonperiodic = training('00110101', csv_path_00110101, epochs=50,
                                    learning_rate=0.1, seed=42, verbose=True)
plot_training(result_len8_nonperiodic, title="DESIRE='00110101' (len=8, non-periodic) | 50 epochs")

In [ ]:
print('='*80)
print('TIER 3: Comparison with previous periodic length-8 run')
print('='*80)

# From colab7 experiment 2: DESIRE='01010101' at 50 epochs
periodic_final = [-0.0216, 1.0017, -0.0165, 1.0, 0.4787, -0.0378, 1.0079, 0.46]
periodic_target = [0, 1, 0, 1, 0, 1, 0, 1]
periodic_naive = max(abs(f - t) for f, t in zip(periodic_final, periodic_target))
periodic_relu = relu_aware_max_err(periodic_final, periodic_target)

print(f"\n{'target':>15} | {'naive_err':>10} | {'relu_err':>10} | {'verdict':>8}")
print('-'*55)
print(f"{'01010101':>15} | {periodic_naive:>10.4f} | {periodic_relu:>10.4f} | {'FAIL'}")
verdict = 'PASS' if result_len8_nonperiodic['relu_pass'] else 'FAIL'
print(f"{'00110101':>15} | {result_len8_nonperiodic['naive_max_err']:>10.4f} | {result_len8_nonperiodic['relu_max_err']:>10.4f} | {verdict}")

print(f"\nWall time: {result_len8_nonperiodic['wall_time']:.1f}s ({result_len8_nonperiodic['wall_time']/3600:.2f}h)")
print(f"\nFinal weights: {[round(w, 4) for w in result_len8_nonperiodic['final_weights']]}")
print(f"Target weights: {result_len8_nonperiodic['target_weights']}")
print(f"Per-position error:")
for i, (f, t) in enumerate(zip(result_len8_nonperiodic['final_weights'],
                                result_len8_nonperiodic['target_weights'])):
    relu_f = max(0.0, f)
    err = abs(relu_f - t)
    marker = '<<<' if err > 0.05 else ''
    print(f'  w_{i+1}: target={t} | final={f:>+.4f} | relu(final)={relu_f:.4f} | err={err:.4f} {marker}')

---
# Final summary

In [ ]:
print('='*90)
print('STRESS TEST 2 SUMMARY')
print('='*90)

print("\n1. ReLU-aware metric on previous DESIRE='01011' results (20 epochs, seeds 0-19):")
n_naive = sum(1 for s in range(20) if max(abs(f - t) for f, t in zip(previous_01011_finals[s], target_01011)) < 0.05)
n_relu = sum(1 for s in range(20) if relu_aware_max_err(previous_01011_finals[s], target_01011) < 0.05)
print(f'   Naive:      {n_naive}/20')
print(f'   ReLU-aware: {n_relu}/20')

print("\n2. Reruns of failed seeds at 50 epochs (DESIRE='01011'):")
n_fixed = sum(1 for s in failed_seeds if rerun_results[s]['relu_pass'])
still_failing = [s for s in failed_seeds if not rerun_results[s]['relu_pass']]
originally_passing = [s for s in range(20) if s not in failed_seeds]
combined_pass = len(originally_passing) + n_fixed
print(f'   {n_fixed}/{len(failed_seeds)} previously failed seeds now pass')
print(f'   Projected total: {combined_pass}/20 pass at 50 epochs (ReLU-aware)')
print(f'   Still failing: {still_failing}')

print("\n3. Partial training reruns at 100 epochs (DESIRE='01011'):")
n_partial_pass = sum(1 for _, _, r in partial_reruns if r['relu_pass'])
print(f'   {n_partial_pass}/{len(partial_reruns)} previously failed (subset, init) combos now pass')
for label, s, r in partial_reruns:
    status = 'PASS' if r['relu_pass'] else 'FAIL'
    print(f'   {label} seed {s}: relu_max_err={r["relu_max_err"]:.4f} | {status}')

print("\n4. Length 8 with non-periodic target '00110101':")
verdict = 'PASS' if result_len8_nonperiodic['relu_pass'] else 'FAIL'
print(f"   relu_max_err = {result_len8_nonperiodic['relu_max_err']:.4f} | {verdict}")
print(f"   wall time: {result_len8_nonperiodic['wall_time']/3600:.2f}h")

print("\nConclusion:")
if result_len8_nonperiodic['relu_pass']:
    print("   - Length 8 works with non-periodic targets")
    print("   - Previous '01010101' failure is attributable to target periodicity")
    print("   - Design B scales structurally; ready to consider DNA generalization")
else:
    print("   - Length 8 fails even with non-periodic targets")
    print("   - Suggests depth-related vanishing gradient is the real blocker")
    print("   - Next step: test optimizer changes (Adam, lr decay) before scaling")